In [1]:
import os
from copy import deepcopy
import json
import time
# import xml.etree.ElementTree as ET
# from rich.tree import Tree
# from rich import print as rprint
import io
from typing import List, Union, Tuple, Dict, Optional
from collections.abc import Iterable
from tqdm import tqdm
# import pdfplumber
# import fitz 
import numpy as np
import pandas as pd
import requests
# import xmltodict
import re
from lxml import etree
from pypdf import PdfReader, PdfWriter
from difflib import SequenceMatcher
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.document import DocumentStream
from docling.pipeline.vlm_pipeline import VlmPipeline
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from table_link_to_excel import _curl_get_text, _extract_pmc_info, _try_pmc_direct_table_download, _flatten_columns, sanitize_sheet_name, fetch_html, fetch_pmc_fulltext_xml, pick_table, table_to_dataframe, _clean_text
from bs4 import BeautifulSoup
from advp_formatting_engine import *
from advp_information_retriever import *
from advp_table_extraction import *
from utils import *

/Users/justpqa/advpai/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Test the pipeline by separating into 2 steps: get the table + get the text col

#### Get the table

In [ ]:
embeddings_model = AutoModel.from_pretrained("NeuML/pubmedbert-base-embeddings")
embeddings_model_tokenizer = AutoTokenizer.from_pretrained("NeuML/pubmedbert-base-embeddings")

In [ ]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"), (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]
# test_papers_info_sample = deepcopy(test_papers_info)
# np.random.seed(37)
# np.random.shuffle(test_papers_info_sample)
# test_papers_info_sample = test_papers_info_sample[:5]
# test_papers_info = [
#     (30820047, "PMC6463297")
# ]
test_papers_info = [
    (20932310, "PMC2964649"), (21123754, "PMC3030225"), (22903471, "PMC3779070"), 
    (23143602, "PMC3510344"), (23535033, "PMC3760995")
]

referencing_col_df = pd.read_csv("referencing_cols/Rules for harmonizing ADVP papers - Main cols.csv")
advp_formatting_engine = ADVPFormattingEngine(referencing_col_df)

referencing_col_require_rag_with_choice_df = pd.read_csv("referencing_cols/ADVP context required col choice.csv")

# NOTE: test results on 3 models and 5 random doc
# 28560309: 7b, 1.5b, 7b, 1.5b, 7b
# 29777097: 7b, 7b, 7b, 7b, 7b
# 30413934: 3b, 3b, 7b, 7b, 7b
# 30448613: 7b, 7b, 7b, 7b, 7b``
# 30979435: 7b, 7b, 7b, 7b, 7b

file_name_to_matching = {}
for pmid, pmcid in test_papers_info:
    print(f"START WORKING WITH {pmid}_{pmcid}")
    # NOTE: find all table id
    found_table = False
    try:
        has_error = table_link_to_excel(pmid, pmcid)
        if has_error:
            print(f"Error in extracting from {pmid}-{pmcid} with table_link_to_excel")
        else:
            found_table = True
            print(f"Success in extracting from {pmid}-{pmcid} with table_link_to_excel")
    except Exception as e:
        print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    # filter for actual GWAS result table: include either SNP (rs....) or CHR, as well as include a p-value cell with p-value col
    curr_intermediate_tables = list(os.listdir("./intermediate_tables"))
    for file_name in curr_intermediate_tables:
        if f"{pmid}_{pmcid}" in file_name:
            if ".xlsx" in file_name:
                df = pd.read_excel(f"./intermediate_tables/{file_name}")
            else:
                df = pd.read_csv(f"./intermediate_tables/{file_name}")
            # check if the table has SNP or CHR, and p value filter
            snp_pattern_1 = re.compile(r"rs\d+")
            snp_pattern_2 = re.compile(r"rsl\d+") # extra pattern of rsl
            snp_mask_1 = df.applymap(lambda x: snp_pattern_1.search(str(x)) is not None)
            snp_mask_2 = df.applymap(lambda x: snp_pattern_2.search(str(x)) is not None)
            snp_filter = snp_mask_1.values.any() or snp_mask_2.values.any()
            chr_pattern_1 = re.compile(r"(?<![A-Za-z.])[Cc][Hh][Rr](?![A-Za-z.])")
            chr_pattern_2 = re.compile('(?<![A-Za-z.])\d+(?![A-Za-z.])')
            chr_mask_1 = df.applymap(lambda x: chr_pattern_1.search(str(x)) is not None)
            chr_mask_2 = df.applymap(lambda x: chr_pattern_2.search(str(x)) is not None)
            chr_filter = chr_mask_1.values.any() and chr_mask_2.values.any()
            p_value_pattern_1 = re.compile(r"\d+\.\d+")
            p_value_pattern_2 = re.compile(r"(?<![A-Za-z])[Pp](?![A-Za-z])",)
            p_value_mask_1 = df.applymap(lambda x: p_value_pattern_1.search(str(x)) is not None)
            p_value_mask_2 = df.applymap(lambda x: p_value_pattern_2.search(str(x)) is not None)
            p_value_filter = p_value_mask_1.values.any() and p_value_mask_2.values.any()
            if (snp_filter or chr_filter) and p_value_filter:
                continue
            else:
                os.remove(f"./intermediate_tables/{file_name}")

    # Temporary remove this part
    # if not found_table:
    #     try: 
    #         df_lst = extract_tables_lst_from_paper(pmcid, f"papers/{pmid}_{pmcid}.pdf")
    #         for i, df in enumerate(df_lst):
    #             if df.shape[0] > 0:
    #                 df.to_csv(f"intermediate_tables/{pmid}_{pmcid}_{i}_from_pdf.csv", index = False)
    #                 found_table = True
    #         if found_table:
    #             print(f"Success in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
    #         else:
    #             print(f"Error in extracting from {pmid}-{pmcid} with extract_tables_lst_from_paper")
    #     except Exception as e:
    #         print(f"Error in extracting from {pmid}-{pmcid} with error {e}")

    try:
        harmonized_df_all = pd.DataFrame(columns = referencing_col_df["column"].to_list())
        for file_name in os.listdir("intermediate_tables"):
            if str(pmid) in file_name and pmcid in file_name:
                if ".xlsx" in file_name:
                    df = pd.read_excel(f"intermediate_tables/{file_name}")
                else:
                    df = pd.read_csv(f"intermediate_tables/{file_name}")

                # save matching dict for debug
                # Conduct cleaning before matching
                clean_df = advp_formatting_engine.clean_df(df)
                file_name_to_matching[file_name] = advp_formatting_engine.match_many_col_to_ref_col(clean_df)

                harmonized_df = advp_formatting_engine.format_original_table(df, remove_unique_col = True)
                if harmonized_df_all is None:
                    harmonized_df_all = harmonized_df.copy()
                else:
                    harmonized_df_all = pd.concat([harmonized_df_all, harmonized_df], ignore_index = True)
        harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
        print(f"Success in harmonizing from {pmid}-{pmcid}")
    except Exception as e:
        print(f"Error in harmonizing from {pmid}-{pmcid} with error {e}")

    # try:
    #     harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
    #     harmonized_df_all["SNP"] = harmonized_df_all["SNP"].apply(lambda x: clean_snp(x))
    #     int_col = ["Chr"]
    #     for c in int_col:
    #         harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_int(x))
    #     numerical_col_with_many_numbers = ["Effect"]
    #     for c in numerical_col_with_many_numbers:
    #         harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: extract_first_number_from_str(x))
    #     float_col = {"P-value": 15, "Effect": None, "AF": None} # effect back to none since in some test it is 3 digits
    #     for c in float_col:
    #         harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_float(x))
    #         if float_col[c] is not None:
    #             harmonized_df_all[c] = harmonized_df_all[c].apply(lambda x: safe_round(x, float_col[c]))
    #     harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
    #     print(f"Success in converting to number from {pmid}-{pmcid}")
    # except Exception as e:
    #     print(f"Error in converting to number from {pmid}-{pmcid} with error {e}")
    # print()

    # NOTE: you need this to prevent 429 error of requesting too much => blocked
    time.sleep(30)

In [ ]:
for f in file_name_to_matching:
    for ref_col in file_name_to_matching[f]:
        for i in range(len(file_name_to_matching[f][ref_col])):
            file_name_to_matching[f][ref_col][i] = (file_name_to_matching[f][ref_col][i][0], float(file_name_to_matching[f][ref_col][i][1]))
with open("test_matching_dict.json", "w") as f:
    json.dump(file_name_to_matching, f, indent=4)

#### Get the text col

In [2]:
# test_papers_info = [
#     (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"), (30617256, "PMC6836675"),
#     (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
#     (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
#     (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
#     (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
# ]
# test_papers_info_sample = deepcopy(test_papers_info)
# np.random.seed(37)
# np.random.shuffle(test_papers_info_sample)
# test_papers_info_sample = test_papers_info_sample[:5]
# test_papers_info = [
#     (30820047, "PMC6463297")
# ]
test_papers_info = [
    (20932310, "PMC2964649"), (21123754, "PMC3030225"), (22903471, "PMC3779070"), 
    (23143602, "PMC3510344"), (23535033, "PMC3760995")
]

referencing_col_require_rag_df = pd.read_csv("referencing_cols/ADVP context required col.csv")
advp_information_retriever = ADVPInformationRetriever(referencing_col_require_rag_df, use_hf = False, device = "mps")

referencing_col_require_rag_with_choice_df = pd.read_csv("referencing_cols/ADVP context required col choice.csv")

# NOTE: test results on 3 models and 5 random doc
# 28560309: 7b, 1.5b, 7b, 1.5b, 7b
# 29777097: 7b, 7b, 7b, 7b, 7b
# 30413934: 3b, 3b, 7b, 7b, 7b
# 30448613: 7b, 7b, 7b, 7b, 7b``
# 30979435: 7b, 7b, 7b, 7b, 7b

file_name_to_matching = {}
for pmid, pmcid in test_papers_info:
    print(pmid, pmcid)
    try:
        # harmonized_df_all = pd.read_csv(f"pred_tables/{pmid}_{pmcid}.csv")
        col_require_rag_to_possible_info = advp_information_retriever.extract_possible_info_from_paper(pmid, pmcid)
    except Exception as e:
        print(f"Error in extracting columns from paper text and LLM from {pmid}-{pmcid} with error {e}")
        col_require_rag_to_possible_info = {col: [] for col in referencing_col_require_rag_df["column"].unique()}
    print(col_require_rag_to_possible_info)
    print()

    # try:
    #     threshold = 0.5
    #     for ref_col, ref_col_choice in zip(
    #         referencing_col_require_rag_with_choice_df["column"], referencing_col_require_rag_with_choice_df["choice"]
    #     ):
    #         if len(col_require_rag_to_possible_info[ref_col]) > 0:
    #             col_with_category = []
    #             if ref_col in col_require_rag_to_possible_info:
    #                 ref_col_choice_lst = ref_col_choice.split(",")
    #                 detail_choice_similarity = calculate_similarity_scores(col_require_rag_to_possible_info[ref_col], ref_col_choice_lst, embeddings_model, embeddings_model_tokenizer)
    #                 # get the max of each col
    #                 max_by_choice = detail_choice_similarity.max(axis = 0).values
    #                 valid_choice = []
    #                 for i in range(len(ref_col_choice_lst)):
    #                     if max_by_choice[i] > threshold:
    #                         valid_choice.append(ref_col_choice_lst[i])
    #                 col_require_rag_to_possible_info[f"{ref_col} category"] = deepcopy(valid_choice)
    #                 col_with_category.append(ref_col)
    #             for ref_col in col_with_category:
    #                 temp, temp_category = col_require_rag_to_possible_info[ref_col], col_require_rag_to_possible_info[f"{ref_col} category"]
    #                 col_require_rag_to_possible_info[ref_col] = deepcopy(temp_category)
    #                 col_require_rag_to_possible_info[f"{ref_col} details"] = deepcopy(temp)
    #                 del col_require_rag_to_possible_info[f"{ref_col} category"]
    #         else:
    #             col_require_rag_to_possible_info[f"{ref_col} details"] = []
    # except Exception as e:
    #     print(f"Error in extracting columns with choice from {pmid}-{pmcid} with error {e}")
    # print(col_require_rag_to_possible_info)
    
    # try:
    #     # for cohort need to do differently
    #     # col_require_rag_to_possible_info["Cohort"] = col_require_rag_to_possible_info["Cohort"] + gwas_information_retriever_cohort.extract_possible_info_from_paper(pmid, pmcid)
    #     # col_require_rag_to_possible_info["Cohort"] = list(set([item.lower() for item in col_require_rag_to_possible_info["Cohort"]]))
    #     harmonized_df_all = match_possible_info_to_df(harmonized_df_all, col_require_rag_to_possible_info, embeddings_model, embeddings_model_tokenizer)
    #     # harmonized_df_all = match_possible_info_to_df_with_clues(harmonized_df_all, pmid, pmcid, gwas_information_retriever)
    #     # harmonized_df_all.to_csv(f"pred_tables/{pmid}_{pmcid}.csv", index = False)
    #     # if (pmid, pmcid) in test_papers_info_sample:
    #     #     harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}_pred.csv", index = False)
    #     harmonized_df_all.to_csv(f"pred_tables_with_text_col/{pmid}_{pmcid}.csv", index = False)
    #     print(f"Success in extracting columns from paper text from {pmid}-{pmcid}")
    # except Exception as e:
    #     print(f"Error in extracting columns from paper text + matching from {pmid}-{pmcid} with error {e}")
    # print()

20932310 PMC2964649
Cohort
METHODS

Methods

Sample

Data used in the preparation of this article were obtained from the Alzheimer's Disease Neuroimaging Initiative (ADNI) database http://www.loni.ucla.edu/ADNI. ADNI was launched in 2003 by the National Institute on Aging (NIA), the National Institute of Biomedical Imaging and Bioengineering (NIBIB), the Food and Drug Administration (FDA), private pharmaceutical companies and non-profit organizations, as a $60 million, 5-year public-private partnership. The primary goal of ADNI has been to test whether serial magnetic resonance imaging (MRI), positron emission tomography (PET), other biological markers, and clinical and neuropsychological assessment can be combined to measure the progression of mild cognitive impairment (MCI) and early Alzheimer's disease (AD). Determination of sensitive and specific markers of very early AD progression is intended to aid researchers and clinicians to develop new treatments and monitor their effectiven

#### Experiment result

- (20932310, "PMC2964649"):
    + Currently estimate many col as P-value due to current pipeline concat all index into 1 single col name, but some col has multi-index that has terms like P-tau but actually are beta terms (for example: Linear regression result without using covariates.5.Normal: A\u03b2 1-42.BETA.Normal: T-tau.Normal: P-tau 181p.MCI: T-tau.MCI: P-tau 181p.AD: A\u03b2 1-42.AD: T-tau.AD: P-tau 181p.Linear regression result using APOE genotype and age as covariates.MCI: A\u03b2 1-42)
    => Have 4x rows than normal
    + Got some extra cohort other than adni like nibib (National Institute of Biomedical Imaging and Bioengineering) or other ways of writing adni, fda, nia
    + Get extra population other than caucasian: 'Multi-ethnic', 'African American', 'Caribbean Hispanic', 'Asian' (based on LLM's: 'race', 'ethnic', 'caucasians', 'ancestry', 'non-caucasian',)
    + Not staged but LLM actually found stage 'meta', 'combined', 'discovery', 'replication'
    + Not found imputation but somehow has ['imputation', 'reference panel', 'pca', 'snp', 'smartpca', 'genotype', 'aligator']
- (21123754, "PMC3030225"):
    + Cannot access due to API limitation
- (22903471, "PMC3779070"):
    + Current consider both R2 and has multiple effect terms (ADNI, QTIM, and Pooled)
    + Consider α as RA
    + We got a lot more rows due to we have 4 different effects (R2 + 2 other groups and pooled), 2 AF (can be of the 2 groups), and 4 p-val (the p-val of 3 groups as above + the p diag, very hard to align these multi-index together of different col)
    + Cohort has extra info, 'discovery', 'study' in addition to adni and qtim
    + Correctly got caucasian, for some reason the matching from raw to certain choice also get african american?
    + Imputation didnt found 1000G, cannot found that in paper too
- (23143602, "PMC3510344"):
    + Extra SE of OR is also considered as effect + along with multiple p-val and multiple effect and multiple AF => a lot of rows
    + Cannot get the ST tables?
    + Only get ADGC cohort and some info related like: "ADGC data were received in three waves of 1763, 1110, and 1266 subjects. In the first wave, 659,224 SNPs were received, while in waves two and three, 730,525 SNPs were received. After QC as described for the **chronic pancreatitis cohort**", "The Stage 2 cohort included 910 cases (331 chronic pancreatitis, 579 recurrent acute pancreatitis; Table 1, Supplementary Table 1), again genotyped at 625,739 SNPs, and 4170 controls, most genotyped previously on the **Illumina 1M**"
    + naps2 got mistaken as a population instead of cohort, Got european population, but also got issue of also getting african american
    + Get too much stage, but also seems like they are close to contain truth? (raw: 'discovery', 'stage 2', 'replication', 'combined', 'gwas', 'meta', 'the joint analysis', 'discoveryreplication', 'stage 1' -> 'Discovery', 'Replication', 'Joint-analysis', 'Meta-analysis', cannot distinguish whihc in which row)
    + Not reported imputation but still found 'gcta software', 'illumina human1m-duo dna analysis beadchip', 'illumina humanomniexpress beadchip', 'plink'
- (23535033, "PMC3760995"):
    + Right cohort, a little bit too extra info (might get from this The strengths of this analysis were the unbiased nature of the GWAS, a discovery and a replication sample, and a statistical model that allowed us to specifically measure test for a differential rate of decline (rather than cognitive function in general) while maximizing the information content of the data (use of repeated measures). Our study was limited by small sample sizes in both datasets, and by the fact that the phenotype of cognitive decline was measured and analyzed differently in the discovery and replication **cohorts**. A full description of these differences is beyond the scope of this paper, but there is face validity to the assumption that both represent a general measure of overall cognitive ability, since both the **ADAS-Cog**, and the **GCOG** incorporate measures on a variety of cognitive domains. Our experience with the ADNI data indicates that the genetic association tests for decline are highly sensitive to the assessment scale used.)
    + Cannot find population from raw info
    + got too much stage info: 'replication', 'discovery', 'meta', might be from The **discovery** sample was 303 AD cases recruited in the AD Neuroimaging Initiative and the replication sample was 323 AD cases from the Religious Orders Study and Rush Memory and Aging Project. In the discovery sample, Alzheimer’s Disease Assessment Scale-cognitive subscale responses were tested for association with genome-wide SNP data using linear regression. We tested the 65 most significant SNPs from the discovery sample for association in the replication sample. and Finally, we **meta-analyzed** the results from the discovery and replication samples using sample size-weighted P-values and the direction of the effect using METAL.25 Associations were considered significant if P values were less than 5 × 10−8.\
    + Right imputation but too much info: 'phasing', 'genotyping', 'imputed genotypes', 'mach', 'reference panel', '1000 genomes reference panel', 'ad', 'eigenstrat'

Try to extract by sections

In [ ]:
unique_section_info = []
url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC2964649/unicode"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    for d in data:
        doc = d["documents"]
        for p in doc:
            passage = p["passages"]
            for item in passage:
                section_info = (item["infons"]["section_type"], item["infons"]["type"])
                if section_info not in unique_section_info:
                    unique_section_info.append(section_info)
unique_section_info

In [ ]:
unique_section_info = []
url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC3779070/unicode"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    for d in data:
        doc = d["documents"]
        for p in doc:
            passage = p["passages"]
            for item in passage:
                section_info = (item["infons"]["section_type"], item["infons"]["type"])
                if section_info not in unique_section_info:
                    unique_section_info.append(section_info)
unique_section_info

In [ ]:
unique_section_info = []
url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC3510344/unicode"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    for d in data:
        doc = d["documents"]
        for p in doc:
            passage = p["passages"]
            for item in passage:
                section_info = (item["infons"]["section_type"], item["infons"]["type"])
                if section_info not in unique_section_info:
                    unique_section_info.append(section_info)
unique_section_info

In [ ]:
unique_section_info = []
url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC3760995/unicode"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    for d in data:
        doc = d["documents"]
        for p in doc:
            passage = p["passages"]
            for item in passage:
                section_info = (item["infons"]["section_type"], item["infons"]["type"])
                if section_info not in unique_section_info:
                    unique_section_info.append(section_info)
unique_section_info

In [ ]:
unique_section_info = []
url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC5237405/unicode"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    for d in data:
        doc = d["documents"]
        for p in doc:
            passage = p["passages"]
            for item in passage:
                section_info = (item["infons"]["section_type"], item["infons"]["type"])
                if section_info not in unique_section_info:
                    unique_section_info.append(section_info)
unique_section_info

In [ ]:
unique_section_info = []
url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC6723529/unicode"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    for d in data:
        doc = d["documents"]
        for p in doc:
            passage = p["passages"]
            for item in passage:
                section_info = (item["infons"]["section_type"], item["infons"]["type"])
                if section_info not in unique_section_info:
                    unique_section_info.append(section_info)
unique_section_info

In [ ]:
test_papers_info = [
    (30448613, "PMC6331247"), (30979435, "PMC6783343"), (28247064, "PMC5613285"), (30617256, "PMC6836675"),
    (30820047, "PMC6463297"), (29458411, "PMC5819208"), (29777097, "PMC5959890"), (30651383, "PMC6369905"),
    (28780673, "PMC5693762"), (30930738, "PMC6425305"), (31426376, "PMC6723529"), (29967939, "PMC6280657"),
    (29107063, "PMC5920782"), (29274321, "PMC5938137"), (30413934, "PMC6358498"), (30805717, "PMC7193309"),
    (30636644, "PMC6330399"), (29752348, "PMC5976227"), (28560309, "PMC5440281"), (27899424, "PMC5237405"),
    (20932310, "PMC2964649"), (21123754, "PMC3030225"), (22903471, "PMC3779070"), (23143602, "PMC3510344"), 
    (23535033, "PMC3760995")
]
unique_section_info = []
for _, pmcid in test_papers_info:
    try:
        url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            for d in data:
                doc = d["documents"]
                for p in doc:
                    passage = p["passages"]
                    for item in passage:
                        section_info = (item["infons"]["section_type"], item["infons"]["type"])
                        if section_info not in unique_section_info:
                            unique_section_info.append(section_info)
    except:
        continue
sorted(unique_section_info)

In [ ]:
sorted(list(set([s[0] for s in unique_section_info])))

In [ ]:
sorted(list(set([s[1] for s in unique_section_info])))